In [60]:
# Define the source directory on Google Drive and target local directory
drive_source_root = Path("/content/drive/MyDrive/Colab Notebooks/GenImage")
LOCAL_GENIMAGE_ROOT = Path("/content/genimage_extracted")

# Safely temporarily point GENIMAGE_ROOT to Drive for extraction
GENIMAGE_ROOT = drive_source_root

# Extract only the necessary splits we need for training (BigGAN), validation (ADM), and testing (Midjourney)
required_extracts = [VALIDATION_GENERATOR, TEST_GENERATOR, REAL_TRAIN_GENERATOR]

for gen in required_extracts:
    extract_multivolume_zip(gen, LOCAL_GENIMAGE_ROOT)

# Now securely point GENIMAGE_ROOT to the fast local path for the training phase
GENIMAGE_ROOT = LOCAL_GENIMAGE_ROOT
print(f"\nUpdated GENIMAGE_ROOT path to fast local storage: {GENIMAGE_ROOT}")

[Skipped] ADM is already extracted at /content/genimage_extracted/ADM
[Skipped] Midjourney is already extracted at /content/genimage_extracted/Midjourney
[Skipped] BigGAN is already extracted at /content/genimage_extracted/BigGAN

Updated GENIMAGE_ROOT path to fast local storage: /content/genimage_extracted


In [39]:
import zipfile
import re
import shutil
from pathlib import Path

def targeted_extract_from_splits(generator_name, split_name, label_folder, max_images, drive_root, local_root):
    """Merges split volumes into a temporary zip stream and extracts only the first max_images."""
    generator_drive_dir = Path(drive_root) / generator_name
    target_local_dir = Path(local_root) / generator_name / split_name / label_folder

    if target_local_dir.exists() and len(list(target_local_dir.glob("*"))) >= max_images:
        print(f"[Skipped] Already extracted {max_images} images for {generator_name}/{split_name}/{label_folder}")
        return

    target_local_dir.mkdir(parents=True, exist_ok=True)

    # Find the split zip parts
    zip_parts = sorted(list(generator_drive_dir.glob("imagenet_*.z*")))
    master_zip = list(generator_drive_dir.glob("imagenet_*.zip"))
    if not master_zip:
         print(f"No zip file found in {generator_drive_dir}")
         return

    print(f"\nPreparing targeted extraction for {generator_name} ({split_name}/{label_folder})...")

    # We combine split files into a temporary zip file
    temp_zip = Path("/tmp") / f"{generator_name}_temp.zip"

    print("Combining split archives virtually...")
    with open(temp_zip, "wb") as outfile:
        for part in zip_parts:
            if part.suffix != ".zip":
                with open(part, "rb") as infile:
                    shutil.copyfileobj(infile, outfile)
        with open(master_zip[0], "rb") as infile:
            shutil.copyfileobj(infile, outfile)

    # Now extract only the needed number of files
    print(f"Extracting up to {max_images} images...")
    count = 0
    target_pattern = re.compile(rf"^{split_name}/{label_folder}/", re.IGNORECASE)

    with zipfile.ZipFile(temp_zip, "r") as z:
        for member in z.infolist():
            if target_pattern.match(member.filename) and not member.is_dir():
                filename = Path(member.filename).name
                dest_file = target_local_dir / filename
                if not dest_file.exists():
                    with z.open(member) as source, open(dest_file, "wb") as target:
                        shutil.copyfileobj(source, target)
                count += 1
                if count >= max_images:
                    break

    # Delete the temporary unified zip file to save space
    temp_zip.unlink(missing_ok=True)
    print(f"[Success] Extracted {count} images to {target_local_dir}")

In [41]:
import subprocess
from pathlib import Path

# Define the source directory on Google Drive and target local directory
drive_root = Path("/content/drive/MyDrive/Colab Notebooks/GenImage")
local_root = Path("/content/genimage_extracted")

def targeted_extract_with_7z(generator, split, label, drive_root, local_root):
    """Extracts ONLY the specific class label subdirectory (e.g. val/nature) using 7z.
    This avoids combining split zip files on disk and prevents No Space Left on Device errors."""
    source_zip = drive_root / generator / f"imagenet_*.zip"
    target_dir = local_root / generator

    # Resolve the wildcard master zip path
    zip_files = list((drive_root / generator).glob("*.zip"))
    if not zip_files:
        print(f"[Error] No master .zip found in {drive_root / generator}")
        return

    master_zip = zip_files[0]
    archive_subfolder = f"{split}/{label}"

    print(f"\nTargeted extraction of {generator} ({archive_subfolder}) directly to local storage...")
    # 7z syntax to extract only a specific folder inside the archive safely:
    # 7z x archive.zip -oOutputDir subfolder/* -r -y
    cmd = ["7z", "x", "-y", str(master_zip), f"-o{target_dir}", f"{archive_subfolder}/*", "-r"]
    try:
        subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(f"[Success] Extracted {archive_subfolder} for {generator}!")
    except subprocess.CalledProcessError as e:
        print(f"[Error] Failed extraction. Error: {e.stderr or e.stdout}")

# Clear previous failed extractions first
import shutil
if local_root.exists():
    shutil.rmtree(local_root)

# Extract ONLY the required folders directly from split zips
targeted_extract_with_7z("ADM", "val", "nature", drive_root, local_root)
targeted_extract_with_7z("ADM", "val", "ai", drive_root, local_root)

targeted_extract_with_7z("Midjourney", "val", "nature", drive_root, local_root)
targeted_extract_with_7z("Midjourney", "val", "ai", drive_root, local_root)

targeted_extract_with_7z("BigGAN", "train", "nature", drive_root, local_root)
targeted_extract_with_7z("BigGAN", "train", "ai", drive_root, local_root)

# Point GENIMAGE_ROOT to our targeted directory
GENIMAGE_ROOT = local_root
print(f"\nTargeted extraction complete! GENIMAGE_ROOT updated to: {GENIMAGE_ROOT}")


Targeted extraction of ADM (val/nature) directly to local storage...
[Success] Extracted val/nature for ADM!

Targeted extraction of ADM (val/ai) directly to local storage...
[Success] Extracted val/ai for ADM!

Targeted extraction of Midjourney (val/nature) directly to local storage...
[Success] Extracted val/nature for Midjourney!

Targeted extraction of Midjourney (val/ai) directly to local storage...
[Success] Extracted val/ai for Midjourney!

Targeted extraction of BigGAN (train/nature) directly to local storage...
[Success] Extracted train/nature for BigGAN!

Targeted extraction of BigGAN (train/ai) directly to local storage...
[Success] Extracted train/ai for BigGAN!

Targeted extraction complete! GENIMAGE_ROOT updated to: /content/genimage_extracted


In [42]:
import zipfile
import re
import shutil
from pathlib import Path

def targeted_extract_from_splits(generator_name, split_name, label_folder, max_images, drive_root, local_root):
    """Merges split volumes into a temporary zip stream and extracts only the first max_images."""
    generator_drive_dir = Path(drive_root) / generator_name
    target_local_dir = Path(local_root) / generator_name / split_name / label_folder

    if target_local_dir.exists() and len(list(target_local_dir.glob("*"))) >= max_images:
        print(f"[Skipped] Already extracted {max_images} images for {generator_name}/{split_name}/{label_folder}")
        return

    target_local_dir.mkdir(parents=True, exist_ok=True)

    # Find the split zip parts
    zip_parts = sorted(list(generator_drive_dir.glob("imagenet_*.z*")))
    master_zip = list(generator_drive_dir.glob("imagenet_*.zip"))
    if not master_zip:
         print(f"No zip file found in {generator_drive_dir}")
         return

    print(f"\nPreparing targeted extraction for {generator_name} ({split_name}/{label_folder})...")

    # We first combine split files into one single virtual zip in /tmp to extract cleanly without taking much space
    temp_zip = Path("/tmp") / f"{generator_name}_temp.zip"

    print("Combining splits virtually...")
    with open(temp_zip, "wb") as outfile:
        for part in zip_parts:
            if part.suffix != ".zip":
                with open(part, "rb") as infile:
                    shutil.copyfileobj(infile, outfile)
        with open(master_zip[0], "rb") as infile:
            shutil.copyfileobj(infile, outfile)

    # Now extract only the needed number of files
    print(f"Extracting up to {max_images} images...")
    count = 0
    # Match files belonging to the requested split and class label (nature or ai)
    # Example path inside zip: train/nature/n01443537_2.JPEG or val/ai/...
    target_pattern = re.compile(rf"^{split_name}/{label_folder}/", re.IGNORECASE)

    with zipfile.ZipFile(temp_zip, "r") as z:
        for member in z.infolist():
            if target_pattern.match(member.filename) and not member.is_dir():
                # Extract file
                filename = Path(member.filename).name
                dest_file = target_local_dir / filename
                if not dest_file.exists():
                    with z.open(member) as source, open(dest_file, "wb") as target:
                        shutil.copyfileobj(source, target)
                count += 1
                if count >= max_images:
                    break

    # Delete the temporary unified zip file to save space
    temp_zip.unlink(missing_ok=True)
    print(f"[Success] Extracted {count} images to {target_local_dir}")

In [44]:
# Perform targeted extraction of exactly what is needed for the notebook splits using 7z
import subprocess
from pathlib import Path
import shutil

drive_root = Path("/content/drive/MyDrive/Colab Notebooks/GenImage")
local_root = Path("/content/genimage_extracted")

def targeted_extract_with_7z(generator, split, label, drive_root, local_root):
    """Extracts ONLY the specific class label subdirectory (e.g. val/nature) using 7z.
    This natively handles multi-volume splits and avoids spanning errors or disk limits."""
    target_dir = local_root / generator

    # Resolve the master zip archive
    zip_files = list((drive_root / generator).glob("*.zip"))
    if not zip_files:
        print(f"[Error] No master .zip found in {drive_root / generator}")
        return

    master_zip = zip_files[0]
    archive_subfolder = f"{split}/{label}"

    print(f"\nTargeted extraction of {generator} ({archive_subfolder}) directly to local storage...")
    # 7z syntax to extract only the specific subfolder directly from the split zip archive:
    cmd = ["7z", "x", "-y", str(master_zip), f"-o{target_dir}", f"{archive_subfolder}/*", "-r"]
    try:
        subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(f"[Success] Extracted {archive_subfolder} for {generator}!")
    except subprocess.CalledProcessError as e:
        print(f"[Error] Failed extraction. Error: {e.stderr or e.stdout}")

# Clear previous failed/partial extractions
if local_root.exists():
    shutil.rmtree(local_root)

# 1. Validation Generator (ADM) -> 5,000 images per class
targeted_extract_with_7z("ADM", "val", "nature", drive_root, local_root)
targeted_extract_with_7z("ADM", "val", "ai", drive_root, local_root)

# 2. Test Generator (Midjourney) -> 5,000 images per class
targeted_extract_with_7z("Midjourney", "val", "nature", drive_root, local_root)
targeted_extract_with_7z("Midjourney", "val", "ai", drive_root, local_root)

# 3. Training Generator (BigGAN) -> 50,000 images per class
targeted_extract_with_7z("BigGAN", "train", "nature", drive_root, local_root)
targeted_extract_with_7z("BigGAN", "train", "ai", drive_root, local_root)

# Point GENIMAGE_ROOT to our target local directory
GENIMAGE_ROOT = local_root
print(f"\nTargeted extraction complete! GENIMAGE_ROOT updated to: {GENIMAGE_ROOT}")


Targeted extraction of ADM (val/nature) directly to local storage...
[Success] Extracted val/nature for ADM!

Targeted extraction of ADM (val/ai) directly to local storage...
[Success] Extracted val/ai for ADM!

Targeted extraction of Midjourney (val/nature) directly to local storage...
[Success] Extracted val/nature for Midjourney!

Targeted extraction of Midjourney (val/ai) directly to local storage...
[Success] Extracted val/ai for Midjourney!

Targeted extraction of BigGAN (train/nature) directly to local storage...
[Success] Extracted train/nature for BigGAN!

Targeted extraction of BigGAN (train/ai) directly to local storage...
[Success] Extracted train/ai for BigGAN!

Targeted extraction complete! GENIMAGE_ROOT updated to: /content/genimage_extracted


# Fine-tune both classifiers on GenImage for AI-generated vs real images

Run in Colab with a GPU. This continues both v2 checkpoints on a balanced GenImage subset: 50,000 real + 50,000 generated training images. Use the official [GenImage shared Drive folder](https://drive.google.com/drive/folders/1jGt10bwTbhEZuGXLyvrCuxOI0cBqQ1FS?usp=sharing); it contains the generator folders used below. The benchmark contains over one million real/fake pairs. See the [GenImage repository](https://github.com/GenImage-Dataset/GenImage) for citation and terms. Review its terms and cite the authors; do not redistribute the dataset. The full dataset is large; this notebook selects only the configured subset.

Evaluation is generator-separated: ADM is validation-only, Midjourney is test-only, and the remaining generators supply training data. Exact byte-identical image duplicates are excluded across splits. Validation guides checkpoint selection; the test generator is not used for training or model selection. The CNN still resizes to 32×32, while the ViT uses its checkpoint processor. Labels are nature/real=0 and AI-generated=1.

**Starting checkpoints:** upload `ai_vs_real_finetuning_outputs_v2.zip`. The notebook writes a separate `ai_vs_real_genimage_outputs_v3` output. A 90% result is not guaranteed: GenImage's published cross-generator benchmark reports roughly 67–70% average accuracy for several standard detectors, so unseen-generator performance is a difficult target. Do not treat accuracy on one held-out generator as proof of universal AI-image detection.

## 1. Install packages and select a device

In [3]:
%pip -q install "transformers>=4.39.3,<5" scikit-learn pandas pillow safetensors

import gc
import hashlib
import json
import os
import random
import shutil
from pathlib import Path
import zipfile

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from transformers import AutoImageProcessor, AutoModelForImageClassification
from google.colab import drive, files

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"TensorFlow: {tf.__version__}; PyTorch: {torch.__version__}")
print(f"TensorFlow GPU(s): {tf.config.list_physical_devices('GPU')}")
print(f"ViT device: {DEVICE}")

TensorFlow: 2.21.0; PyTorch: 2.14.1+cu130
TensorFlow GPU(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
ViT device: cuda


## 2. Mount GenImage from Drive and load the v2 checkpoints

This folder is shared by the dataset owner; it is not your Drive, and you do not need to own it. Sign into an account that has access. Add a shortcut to the owner's shared folder under your `My Drive/GenImage` so Colab can see it after Drive is mounted. A shortcut does not copy or transfer ownership. The official GenImage instructions say the generator folders contain compressed archives; if the shortcut does not expose extracted `train/ai`, `train/nature`, `val/ai`, and `val/nature` folders, copy/download and extract the needed archives into your own Drive. That can require substantial storage and time. The notebook expects `MyDrive/GenImage/BigGAN`, `VQDM`, `ADM`, and `Midjourney`, each with those four subfolders. The shared folder lists `ADM`, `BigGAN`, `glide`, `Midjourney`, `stable_diffusion_v_1_4`, `stable_diffusion_v_1_5`, `VQDM`, and `wukong`. If your shortcut or extracted folder is elsewhere, change `GENIMAGE_ROOT` below. Upload `ai_vs_real_finetuning_outputs_v2.zip` when prompted; it must contain the v2 CNN and ViT weights/processor.

In [18]:
from pathlib import PurePosixPath

drive.mount("/content/drive")
GENIMAGE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/GenImage")
if not GENIMAGE_ROOT.is_dir():
    raise FileNotFoundError(
        f"GenImage root not found at {GENIMAGE_ROOT}. Add a Drive shortcut to the "
        "owner-shared folder or extract the required generator archives into your Drive, "
        "then update GENIMAGE_ROOT."
    )
print(f"GenImage root: {GENIMAGE_ROOT}")

PREVIOUS_OUTPUTS_ZIP = Path("/content/ai_vs_real_finetuning_outputs_v2.zip")
PREVIOUS_OUTPUTS_DIR = Path("/content/previous_ai_vs_real_finetuning_outputs_v2")
CNN_CHECKPOINT = PREVIOUS_OUTPUTS_DIR / "cnn" / "cifake_custom_cnn_ai_vs_real_further_finetuned.keras"
VIT_CHECKPOINT_DIR = PREVIOUS_OUTPUTS_DIR / "huggingface_vit"

required_previous_files = (
    CNN_CHECKPOINT,
    VIT_CHECKPOINT_DIR / "config.json",
    VIT_CHECKPOINT_DIR / "model.safetensors",
    VIT_CHECKPOINT_DIR / "preprocessor_config.json",
)
if not all(path.is_file() for path in required_previous_files):
    if not PREVIOUS_OUTPUTS_ZIP.is_file():
        print("Upload the previous ai_vs_real_finetuning_outputs_v2.zip from your computer.")
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(zip_names) != 1:
            raise ValueError(f"Upload exactly one previous fine-tuning ZIP; received {list(uploaded)}")
        PREVIOUS_OUTPUTS_ZIP = Path("/content") / zip_names[0]

    PREVIOUS_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PREVIOUS_OUTPUTS_ZIP) as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue
            member_path = PurePosixPath(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in previous-output ZIP: {member.filename}")
            parts = member_path.parts
            model_root_index = next(
                (index for index, part in enumerate(parts) if part in {"cnn", "huggingface_vit"}),
                None,
            )
            if model_root_index is None:
                continue
            relative_path = Path(*parts[model_root_index:])
            target = (PREVIOUS_OUTPUTS_DIR / relative_path).resolve()
            if not target.is_relative_to(PREVIOUS_OUTPUTS_DIR.resolve()):
                raise ValueError(f"Unsafe output path in ZIP: {member.filename}")
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)

missing_files = [str(path) for path in required_previous_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "The previous ZIP is missing required continuation checkpoints: "
        + ", ".join(missing_files)
    )
print(f"Continuing CNN from: {CNN_CHECKPOINT}")
print(f"Continuing ViT and processor from: {VIT_CHECKPOINT_DIR}")
print(f"Continuing from v2; new checkpoints will be written separately.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GenImage root: /content/drive/MyDrive/Colab Notebooks/GenImage
Upload the previous ai_vs_real_finetuning_outputs_v2.zip from your computer.


Saving ai_vs_real_finetuning_outputs_v2.zip to ai_vs_real_finetuning_outputs_v2.zip
Continuing CNN from: /content/previous_ai_vs_real_finetuning_outputs_v2/cnn/cifake_custom_cnn_ai_vs_real_further_finetuned.keras
Continuing ViT and processor from: /content/previous_ai_vs_real_finetuning_outputs_v2/huggingface_vit
Continuing from v2; new checkpoints will be written separately.


## 3. Build balanced splits with generator-held-out evaluation

The notebook uses 50,000 training images per class, 5,000 validation images per class from ADM, and 5,000 test images per class from Midjourney. The test generator is excluded from training and validation. Real training images come from one training generator's `nature` folder; exact SHA-256 duplicates are removed across all splits to prevent direct image leakage.

In [76]:
import os
import hashlib
from pathlib import Path

import pandas as pd

SEED = 42
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
TRAIN_PER_CLASS = 50_000
VALIDATION_PER_CLASS = 5_000
TEST_PER_CLASS = 5_000
VALIDATION_GENERATOR = "ADM"
TEST_GENERATOR = "Midjourney"
REAL_TRAIN_GENERATOR = "BigGAN"
CLASS_FOLDERS = {0: "nature", 1: "ai"}

GENIMAGE_ROOT = Path("/content/genimage_extracted")

# ---------- Generator discovery ----------
generator_roots = {p.name: p for p in GENIMAGE_ROOT.iterdir() if p.is_dir()}
required_generators = {VALIDATION_GENERATOR, TEST_GENERATOR, REAL_TRAIN_GENERATOR}
missing_generators = required_generators - set(generator_roots)
if missing_generators:
    raise FileNotFoundError(
        f"Missing required GenImage generator folders: {sorted(missing_generators)}. "
        f"Found: {sorted(generator_roots)}"
    )

TRAINING_GENERATORS = sorted(set(generator_roots) - {VALIDATION_GENERATOR, TEST_GENERATOR})
if REAL_TRAIN_GENERATOR not in TRAINING_GENERATORS:
    raise ValueError("REAL_TRAIN_GENERATOR must be included among training generators.")
if VALIDATION_GENERATOR == TEST_GENERATOR:
    raise ValueError("Validation and test generators must be different.")

print(f"Training generators: {TRAINING_GENERATORS}")
print(f"Validation generator: {VALIDATION_GENERATOR}; test generator: {TEST_GENERATOR}")


# ---------- Candidate collection (strict split matching) ----------
def collect_candidates(generators, split_name, label):
    folder = CLASS_FOLDERS[label]
    records = []
    for generator in generators:
        gen_dir = generator_roots[generator]

        found_paths = [p for p in gen_dir.rglob(folder) if p.is_dir()]
        # Strict: the path must contain the requested split ("train" or "val").
        # No fallback, so train images can never leak into val/test.
        matches = [p for p in found_paths if split_name in p.relative_to(gen_dir).parts]
        if not matches:
            raise FileNotFoundError(
                f"No '{split_name}/{folder}' for generator '{generator}' in {gen_dir}. "
                f"Found: {[str(p.relative_to(gen_dir)) for p in found_paths]}"
            )
        class_dir = min(matches, key=lambda p: len(p.parts))  # shallowest match

        print(f"Reading images from {generator} ({split_name}/{folder}) -> {class_dir}")
        records.extend(
            {"path": str(path), "label": label, "generator": generator}
            for path in class_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
    if not records:
        raise ValueError(f"No images found for generators={generators}, split={split_name}, label={label}")
    return pd.DataFrame(records).drop_duplicates(subset="path").reset_index(drop=True)


def candidates_for(generators, source_split, label):
    frame = collect_candidates(generators, source_split, label)
    frame["source_split"] = source_split
    return frame


# ---------- Hash-based dedup across all splits ----------
seen_hashes = {}


def file_sha256(path):
    digest = hashlib.sha256()
    try:
        with Path(path).open("rb") as image_file:
            for chunk in iter(lambda: image_file.read(1024 * 1024), b""):
                digest.update(chunk)
    except OSError as error:
        raise RuntimeError(f"Cannot hash GenImage image {path}: {error}") from error
    return digest.hexdigest()


def select_unique(records, count, split_name):
    shuffled = records.sample(frac=1, random_state=SEED).reset_index(drop=True)
    selected = []
    for row in shuffled.itertuples(index=False):
        digest = file_sha256(row.path)
        prior = seen_hashes.get(digest)
        if prior is not None:
            if prior[0] != int(row.label):
                raise ValueError(f"Identical image content has conflicting labels: {row.path}")
            continue
        seen_hashes[digest] = (int(row.label), split_name)
        selected.append({
            "path": row.path,
            "label": int(row.label),
            "generator": row.generator,
            "source_split": row.source_split,
        })
        if len(selected) == count:
            break
    if len(selected) != count:
        raise ValueError(
            f"Needed {count:,} unique label-{int(records['label'].iloc[0])} images for "
            f"{split_name}, but found only {len(selected):,}. "
            f"Lower the per-class count or add more GenImage files."
        )
    return pd.DataFrame(selected)


# ---------- Build splits ----------
selected_test = [
    select_unique(candidates_for([TEST_GENERATOR], "val", label), TEST_PER_CLASS, "test")
    for label in (0, 1)
]
selected_validation = [
    select_unique(candidates_for([VALIDATION_GENERATOR], "val", label), VALIDATION_PER_CLASS, "validation")
    for label in (0, 1)
]
selected_train = []
for label in (0, 1):
    source_generators = [REAL_TRAIN_GENERATOR] if label == 0 else TRAINING_GENERATORS
    selected_train.append(
        select_unique(candidates_for(source_generators, "train", label), TRAIN_PER_CLASS, "train")
    )

train_frame = (
    pd.concat(selected_train, ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
validation_frame = pd.concat(selected_validation, ignore_index=True).reset_index(drop=True)
test_frame = pd.concat(selected_test, ignore_index=True).reset_index(drop=True)

for split_name, split_frame in (("train", train_frame), ("validation", validation_frame), ("test", test_frame)):
    counts = split_frame["label"].value_counts().sort_index().to_dict()
    print(f"{split_name}: {len(split_frame):,} unique images; labels={counts}")

# ---------- Sanity checks ----------
assert set(train_frame["label"].unique()) == {0, 1}
assert set(validation_frame["label"].unique()) == {0, 1}
assert set(test_frame["label"].unique()) == {0, 1}
assert not (set(train_frame.path) & set(validation_frame.path))
assert not (set(train_frame.path) & set(test_frame.path))
assert not (set(validation_frame.path) & set(test_frame.path))
assert (train_frame["source_split"] == "train").all()
assert (validation_frame["source_split"] == "val").all()
assert (test_frame["source_split"] == "val").all()
print("All split checks passed.")

Training generators: ['BigGAN']
Validation generator: ADM; test generator: Midjourney
Reading images from Midjourney (val/nature) -> /content/genimage_extracted/Midjourney/imagenet_midjourney/val/nature
Reading images from Midjourney (val/ai) -> /content/genimage_extracted/Midjourney/imagenet_midjourney/val/ai
Reading images from ADM (val/nature) -> /content/genimage_extracted/ADM/imagenet_ai_0508_adm/val/nature
Reading images from ADM (val/ai) -> /content/genimage_extracted/ADM/imagenet_ai_0508_adm/val/ai
Reading images from BigGAN (train/nature) -> /content/genimage_extracted/BigGAN/imagenet_ai_0419_biggan/train/nature
Reading images from BigGAN (train/ai) -> /content/genimage_extracted/BigGAN/imagenet_ai_0419_biggan/train/ai
train: 100,000 unique images; labels={0: 50000, 1: 50000}
validation: 10,000 unique images; labels={0: 5000, 1: 5000}
test: 10,000 unique images; labels={0: 5000, 1: 5000}
All split checks passed.


In [21]:
def print_directory_tree(start_path, max_depth=3, current_depth=0):
    """Recursively prints the directory structure of a given path up to max_depth."""
    path = Path(start_path)
    if not path.exists():
        print(f"Path does not exist: {start_path}")
        return

    indent = "  " * current_depth
    if path.is_dir():
        print(f"{indent}📁 {path.name}/")
        if current_depth >= max_depth:
            return
        try:
            for item in sorted(path.iterdir()):
                # Skip hidden files or OS artifacts
                if item.name.startswith('.'):
                    continue
                print_directory_tree(item, max_depth, current_depth + 1)
        except PermissionError:
            print(f"{indent}  [Permission Denied]")
    else:
        # Optional: Print file names (truncated if needed)
        print(f"{indent}📄 {path.name}")

# Run the verification on your GenImage directory up to depth 3
print_directory_tree(GENIMAGE_ROOT, max_depth=3)

📁 GenImage/
  📁 ADM/
    📄 imagenet_ai_0508_adm.z01
    📄 imagenet_ai_0508_adm.z02
    📄 imagenet_ai_0508_adm.z03
    📄 imagenet_ai_0508_adm.z04
    📄 imagenet_ai_0508_adm.z05
    📄 imagenet_ai_0508_adm.z06
    📄 imagenet_ai_0508_adm.z07
    📄 imagenet_ai_0508_adm.z08
    📄 imagenet_ai_0508_adm.z09
    📄 imagenet_ai_0508_adm.z10
    📄 imagenet_ai_0508_adm.z11
    📄 imagenet_ai_0508_adm.z12
    📄 imagenet_ai_0508_adm.zip
  📁 BigGAN/
    📄 imagenet_ai_0419_biggan.z01
    📄 imagenet_ai_0419_biggan.z02
    📄 imagenet_ai_0419_biggan.z03
    📄 imagenet_ai_0419_biggan.z04
    📄 imagenet_ai_0419_biggan.z05
    📄 imagenet_ai_0419_biggan.z06
    📄 imagenet_ai_0419_biggan.z07
    📄 imagenet_ai_0419_biggan.zip
  📁 Midjourney/
    📄 imagenet_midjourney.z01
    📄 imagenet_midjourney.z02
    📄 imagenet_midjourney.z03
    📄 imagenet_midjourney.z04
    📄 imagenet_midjourney.z05
    📄 imagenet_midjourney.z06
    📄 imagenet_midjourney.z07
    📄 imagenet_midjourney.z08
    📄 imagenet_midjourney.z09
    📄 

### Extraction Helper
Because GenImage datasets are provided in multi-volume `.zip` splits (e.g., `.z01`, `.z02`, ... and `.zip`), we will use the system `unzip` utility (which natively handles multi-volume combined zip streams) to extract the required subfolders into the local fast runtime storage (`/content/genimage_extracted`).

In [26]:
import subprocess
from pathlib import Path

# Ensure 7-Zip is installed (usually pre-installed, but let's make sure)
subprocess.run(["apt-get", "install", "-y", "p7zip-full"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def extract_multivolume_zip(generator_name, target_extracted_root):
    """Combines and extracts multi-volume zip split archives to local storage using 7z."""
    generator_drive_path = GENIMAGE_ROOT / generator_name
    target_dir = Path(target_extracted_root) / generator_name

    if target_dir.exists() and any(target_dir.iterdir()):
        print(f"[Skipped] {generator_name} is already extracted at {target_dir}")
        return

    # Find the main .zip file inside the generator directory
    zip_files = list(generator_drive_path.glob("*.zip"))
    if not zip_files:
        print(f"[Error] No master .zip file found in {generator_drive_path}")
        return

    master_zip = zip_files[0]
    target_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nExtracting {generator_name} using 7z from {master_zip} to {target_dir}...")
    print("This will automatically merge split volumes and extract them. Please wait...")

    # -o{path} sets output directory. Note: space is NOT allowed between -o and the path in 7z
    cmd = ["7z", "x", "-y", str(master_zip), f"-o{target_dir}"]
    try:
        # We run without -q since 7z output is cleaner, but redirect stdout to clean up the screen
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(f"[Success] Finished extracting {generator_name}!")
    except subprocess.CalledProcessError as e:
        print(f"[Error] Failed extracting {generator_name}. Error: {e.stderr or e.stdout}")

In [27]:
# We will define our local extraction root directory
LOCAL_GENIMAGE_ROOT = Path("/content/genimage_extracted")

# Extract only the necessary splits we need for training, validation, and testing
required_extracts = [VALIDATION_GENERATOR, TEST_GENERATOR, REAL_TRAIN_GENERATOR]

for gen in required_extracts:
    extract_multivolume_zip(gen, LOCAL_GENIMAGE_ROOT)

# Update our GENIMAGE_ROOT pointer to the fast local path
GENIMAGE_ROOT = LOCAL_GENIMAGE_ROOT
print(f"\nUpdated GENIMAGE_ROOT path to fast local storage: {GENIMAGE_ROOT}")

[Error] No master .zip file found in /content/genimage_extracted/ADM
[Error] No master .zip file found in /content/genimage_extracted/Midjourney
[Skipped] BigGAN is already extracted at /content/genimage_extracted/BigGAN

Updated GENIMAGE_ROOT path to fast local storage: /content/genimage_extracted


In [49]:
# Explicitly extract the remaining targeted folders for Midjourney and BigGAN
targeted_extract_with_7z("Midjourney", "val", "nature", drive_root, local_root)
targeted_extract_with_7z("Midjourney", "val", "ai", drive_root, local_root)

targeted_extract_with_7z("BigGAN", "train", "nature", drive_root, local_root)
targeted_extract_with_7z("BigGAN", "train", "ai", drive_root, local_root)

print(f"\nRemaining extractions complete! Local dataset directory verified at: {GENIMAGE_ROOT}")


Targeted extraction of Midjourney (val/nature) directly to local storage...
[Success] Extracted val/nature for Midjourney!

Targeted extraction of Midjourney (val/ai) directly to local storage...
[Success] Extracted val/ai for Midjourney!

Targeted extraction of BigGAN (train/nature) directly to local storage...
[Success] Extracted train/nature for BigGAN!

Targeted extraction of BigGAN (train/ai) directly to local storage...
[Success] Extracted train/ai for BigGAN!

Remaining extractions complete! Local dataset directory verified at: /content/drive/MyDrive/Colab Notebooks/GenImage


## 4. Continue fine-tuning the existing CIFAKE Custom CNN

This starts from the previous AI-vs-real fine-tuned `.keras` checkpoint. It evaluates that starting model on validation data, continues training at a low learning rate, then keeps whichever checkpoint has lower validation loss. The original checkpoint is never overwritten.

In [1]:
OUTPUT_DIR = Path("/content/ai_vs_real_genimage_outputs_v3")
CNN_OUTPUT_DIR = OUTPUT_DIR / "cnn"
VIT_OUTPUT_DIR = OUTPUT_DIR / "huggingface_vit"
METRICS_DIR = OUTPUT_DIR / "metrics"
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(
        f"Output directory is not empty: {OUTPUT_DIR}. Download its contents or "
        "use a fresh Colab runtime before rerunning to avoid mixing checkpoints."
    )
for directory in (CNN_OUTPUT_DIR, VIT_OUTPUT_DIR, METRICS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CNN_SIZE = (32, 32)
CNN_BATCH_SIZE = 64
CNN_EPOCHS = 8
CNN_LEARNING_RATE = 1e-5
AUTOTUNE = tf.data.AUTOTUNE

def decode_cnn_image(path_tensor):
    def load_one(path_value):
        path = path_value.decode("utf-8")
        try:
            with Image.open(path) as image:
                image = image.convert("RGB").resize(CNN_SIZE, Image.Resampling.BILINEAR)
                return np.asarray(image, dtype=np.uint8)
        except (OSError, UnidentifiedImageError) as error:
            raise RuntimeError(f"Cannot decode image {path}: {error}") from error
    tensor = tf.numpy_function(load_one, [path_tensor], tf.uint8)
    tensor.set_shape((32, 32, 3))
    return tf.cast(tensor, tf.float32) / 255.0

def make_cnn_dataset(split_frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        split_frame["path"].astype(str).to_numpy(),
        split_frame["label"].to_numpy(dtype=np.float32),
    ))
    dataset = dataset.map(
        lambda path, label: (decode_cnn_image(path), label),
        num_parallel_calls=AUTOTUNE,
    )
    if training:
        dataset = dataset.shuffle(len(split_frame), seed=SEED, reshuffle_each_iteration=True)
    return dataset.batch(CNN_BATCH_SIZE).prefetch(AUTOTUNE)

cnn_model = tf.keras.models.load_model(CNN_CHECKPOINT, compile=False)
if tuple(cnn_model.input_shape[1:]) != (32, 32, 3):
    raise ValueError(f"Expected a CIFAKE (32,32,3) CNN checkpoint; got {cnn_model.input_shape}")
if cnn_model.output_shape[-1] != 1:
    raise ValueError(f"Expected a single sigmoid output; got {cnn_model.output_shape}")
def compile_cnn(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=CNN_LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")],
    )
    return model

cnn_model = compile_cnn(cnn_model)
cnn_validation_dataset = make_cnn_dataset(validation_frame)
starting_cnn_val_loss = float(cnn_model.evaluate(cnn_validation_dataset, verbose=0)[0])
print(f"Starting CNN validation loss: {starting_cnn_val_loss:.5f}")
cnn_best_path = CNN_OUTPUT_DIR / "cifake_custom_cnn_ai_vs_real_candidate.keras"
cnn_history = cnn_model.fit(
    make_cnn_dataset(train_frame, training=True),
    validation_data=cnn_validation_dataset,
    epochs=CNN_EPOCHS,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            cnn_best_path, monitor="val_loss", mode="min", save_best_only=True
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", mode="min", patience=2, restore_best_weights=True
        ),
    ],
    verbose=1,
)
candidate_cnn = compile_cnn(tf.keras.models.load_model(cnn_best_path, compile=False))
candidate_cnn_val_loss = float(candidate_cnn.evaluate(cnn_validation_dataset, verbose=0)[0])
if candidate_cnn_val_loss <= starting_cnn_val_loss:
    cnn_model = candidate_cnn
    print(f"Using continued CNN; validation loss {candidate_cnn_val_loss:.5f} <= {starting_cnn_val_loss:.5f}")
else:
    del candidate_cnn
    cnn_model = tf.keras.models.load_model(CNN_CHECKPOINT, compile=False)
    print(f"Keeping starting CNN; candidate validation loss {candidate_cnn_val_loss:.5f} > {starting_cnn_val_loss:.5f}")
cnn_scores = np.concatenate([
    np.asarray(cnn_model(images, training=False)).reshape(-1)
    for images, _ in make_cnn_dataset(test_frame)
])
cnn_true = test_frame["label"].to_numpy(dtype=np.int64)
cnn_model.save(CNN_OUTPUT_DIR / "cifake_custom_cnn_ai_vs_real_genimage_v3.keras")
print(f"Saved selected CNN: {CNN_OUTPUT_DIR / 'cifake_custom_cnn_ai_vs_real_genimage_v3.keras'}")
cnn_selection = {
    "starting_validation_loss": starting_cnn_val_loss,
    "continued_validation_loss": candidate_cnn_val_loss,
    "continued_checkpoint_selected": candidate_cnn_val_loss <= starting_cnn_val_loss,
}
del cnn_model
tf.keras.backend.clear_session()
gc.collect()

NameError: name 'Path' is not defined

## 5. Continue fine-tuning the Hugging Face ViT

The ViT and processor load from the previous run's uploaded output ZIP, not the original Hugging Face checkpoint. Training preserves the `REAL`/`FAKE` class mapping. The previous checkpoint is retained if continued training does not improve validation accuracy (with validation loss as a tie-breaker).

In [ ]:
MODEL_ID = "dima806/ai_vs_real_image_detection"
MODEL_REVISION = "5d647683672ffd0080441f4e8b4345934c44cc61"
VIT_BATCH_SIZE = 16
VIT_EPOCHS = 3
VIT_LEARNING_RATE = 2e-6

processor = AutoImageProcessor.from_pretrained(
    str(VIT_CHECKPOINT_DIR), local_files_only=True
)
vit_model = AutoModelForImageClassification.from_pretrained(
    str(VIT_CHECKPOINT_DIR), local_files_only=True, use_safetensors=True
)
id2label = {
    int(index): str(label).strip().upper()
    for index, label in vit_model.config.id2label.items()
}
real_ids = [index for index, label in id2label.items() if label == "REAL"]
fake_ids = [index for index, label in id2label.items() if label == "FAKE"]
if len(real_ids) != 1 or len(fake_ids) != 1:
    raise ValueError(f"Expected one REAL and one FAKE output class; got {id2label}")
REAL_CLASS_ID, FAKE_CLASS_ID = real_ids[0], fake_ids[0]
label2class_id = {0: REAL_CLASS_ID, 1: FAKE_CLASS_ID}
print(f"ViT labels: {id2label}; training target mapping: {label2class_id}")

class AIRealImageDataset(torch.utils.data.Dataset):
    def __init__(self, split_frame):
        self.paths = split_frame["path"].tolist()
        self.labels = split_frame["label"].to_numpy(dtype=np.int64)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image_path = self.paths[index]
        try:
            with Image.open(image_path) as image:
                inputs = processor(images=image.convert("RGB"), return_tensors="pt")
        except (OSError, UnidentifiedImageError) as error:
            raise RuntimeError(f"Cannot decode image {image_path}: {error}") from error
        label = label2class_id[int(self.labels[index])]
        return inputs["pixel_values"].squeeze(0), label

def collate_face_batch(batch):
    pixel_values, labels = zip(*batch)
    return torch.stack(pixel_values), torch.tensor(labels, dtype=torch.long)

train_loader = torch.utils.data.DataLoader(
    AIRealImageDataset(train_frame), batch_size=VIT_BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=torch.cuda.is_available(), collate_fn=collate_face_batch,
)
validation_loader = torch.utils.data.DataLoader(
    AIRealImageDataset(validation_frame), batch_size=VIT_BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available(), collate_fn=collate_face_batch,
)
test_loader = torch.utils.data.DataLoader(
    AIRealImageDataset(test_frame), batch_size=VIT_BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available(), collate_fn=collate_face_batch,
)

vit_model.to(DEVICE)
optimizer = torch.optim.AdamW(vit_model.parameters(), lr=VIT_LEARNING_RATE)

def evaluate_vit_validation(model):
    model.eval()
    labels, scores = [], []
    loss_sum = 0.0
    sample_count = 0
    with torch.inference_mode():
        for pixel_values, class_ids in validation_loader:
            pixel_values = pixel_values.to(DEVICE, non_blocking=True)
            device_class_ids = class_ids.to(DEVICE, non_blocking=True)
            output = model(pixel_values=pixel_values, labels=device_class_ids)
            probabilities = torch.softmax(output.logits, dim=-1)[:, FAKE_CLASS_ID]
            labels.extend(0 if int(value) == REAL_CLASS_ID else 1 for value in class_ids)
            scores.extend(probabilities.cpu().tolist())
            loss_sum += float(output.loss.item()) * len(class_ids)
            sample_count += len(class_ids)
    predicted = (np.asarray(scores) >= 0.5).astype(np.int64)
    return float(accuracy_score(labels, predicted)), loss_sum / sample_count

starting_vit_val_accuracy, starting_vit_val_loss = evaluate_vit_validation(vit_model)
print(
    f"Starting ViT validation accuracy={starting_vit_val_accuracy:.2%}, "
    f"loss={starting_vit_val_loss:.5f}"
)
best_val_accuracy = starting_vit_val_accuracy
best_val_loss = starting_vit_val_loss
epochs_without_improvement = 0
vit_continued_selected = False
vit_history = []
for epoch in range(VIT_EPOCHS):
    vit_model.train()
    train_loss_sum = 0.0
    train_count = 0
    for pixel_values, class_ids in train_loader:
        pixel_values = pixel_values.to(DEVICE, non_blocking=True)
        class_ids = class_ids.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        output = vit_model(pixel_values=pixel_values, labels=class_ids)
        output.loss.backward()
        optimizer.step()
        train_loss_sum += float(output.loss.item()) * len(class_ids)
        train_count += len(class_ids)

    vit_model.eval()
    validation_true, validation_scores = [], []
    validation_loss_sum = 0.0
    validation_count = 0
    with torch.inference_mode():
        for pixel_values, class_ids in validation_loader:
            device_pixel_values = pixel_values.to(DEVICE, non_blocking=True)
            device_class_ids = class_ids.to(DEVICE, non_blocking=True)
            output = vit_model(pixel_values=device_pixel_values, labels=device_class_ids)
            fake_probabilities = torch.softmax(output.logits, dim=-1)[:, FAKE_CLASS_ID]
            semantic_labels = [0 if int(value) == REAL_CLASS_ID else 1 for value in class_ids]
            validation_true.extend(semantic_labels)
            validation_scores.extend(fake_probabilities.cpu().tolist())
            validation_loss_sum += float(output.loss.item()) * len(class_ids)
            validation_count += len(class_ids)
    val_predictions = (np.asarray(validation_scores) >= 0.5).astype(np.int64)
    val_accuracy = float(accuracy_score(validation_true, val_predictions))
    val_loss = validation_loss_sum / validation_count
    epoch_record = {
        "epoch": epoch + 1,
        "train_loss": train_loss_sum / train_count,
        "validation_accuracy": val_accuracy,
        "validation_loss": val_loss,
    }
    vit_history.append(epoch_record)
    print(epoch_record)
    if val_accuracy > best_val_accuracy or (
        val_accuracy == best_val_accuracy and val_loss < best_val_loss
    ):
        best_val_accuracy = val_accuracy
        best_val_loss = val_loss
        vit_continued_selected = True
        epochs_without_improvement = 0
        vit_model.save_pretrained(VIT_OUTPUT_DIR, safe_serialization=True)
        processor.save_pretrained(VIT_OUTPUT_DIR)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= 2:
            print("Early stopping: validation accuracy did not improve.")
            break

if not vit_continued_selected:
    print("Continued ViT did not improve validation; retaining the starting checkpoint.")
    vit_model = AutoModelForImageClassification.from_pretrained(
        str(VIT_CHECKPOINT_DIR), local_files_only=True, use_safetensors=True
    )
    vit_model.save_pretrained(VIT_OUTPUT_DIR, safe_serialization=True)
    processor.save_pretrained(VIT_OUTPUT_DIR)
with (METRICS_DIR / "huggingface_vit_training_history.json").open("w", encoding="utf-8") as file:
    json.dump(vit_history, file, indent=2)
vit_selection = {
    "starting_validation_accuracy": starting_vit_val_accuracy,
    "starting_validation_loss": starting_vit_val_loss,
    "selected_validation_accuracy": best_val_accuracy,
    "selected_validation_loss": best_val_loss,
    "continued_checkpoint_selected": vit_continued_selected,
}
vit_model = AutoModelForImageClassification.from_pretrained(
    VIT_OUTPUT_DIR, local_files_only=True, use_safetensors=True
).to(DEVICE).eval()
vit_test_true, vit_test_scores = [], []
with torch.inference_mode():
    for pixel_values, class_ids in test_loader:
        logits = vit_model(pixel_values=pixel_values.to(DEVICE, non_blocking=True)).logits
        fake_probabilities = torch.softmax(logits, dim=-1)[:, FAKE_CLASS_ID]
        semantic_labels = [0 if int(value) == REAL_CLASS_ID else 1 for value in class_ids]
        vit_test_true.extend(semantic_labels)
        vit_test_scores.extend(fake_probabilities.cpu().tolist())
print(f"Saved best validation-selected ViT and processor to {VIT_OUTPUT_DIR}")

## 6. Evaluate both fine-tuned models on the same held-out test images

In [ ]:
def binary_metrics(y_true, ai_scores):
    y_true = np.asarray(y_true, dtype=np.int64)
    ai_scores = np.asarray(ai_scores, dtype=np.float64)
    if len(y_true) != len(ai_scores) or not np.isfinite(ai_scores).all():
        raise ValueError("Invalid labels or model scores.")
    predicted = (ai_scores >= 0.5).astype(np.int64)
    return {
        "n": int(len(y_true)),
        "real_count": int(np.sum(y_true == 0)),
        "ai_generated_count": int(np.sum(y_true == 1)),
        "threshold": 0.5,
        "accuracy": float(accuracy_score(y_true, predicted)),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "f1_score": float(f1_score(y_true, predicted, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, ai_scores)),
        "confusion_matrix_labels_0_1": confusion_matrix(y_true, predicted, labels=[0, 1]).tolist(),
    }

cnn_results = binary_metrics(cnn_true, cnn_scores)
vit_results = binary_metrics(vit_test_true, vit_test_scores)
if cnn_results["n"] != vit_results["n"] or cnn_results["n"] != len(test_frame):
    raise RuntimeError("Both models must be evaluated on the same complete test split.")

comparison = pd.DataFrame([
    {"model": "Further-fine-tuned CIFAKE CNN", **cnn_results},
    {"model": "Further-fine-tuned Hugging Face ViT", **vit_results},
])
display(comparison[["model", "n", "accuracy", "precision", "recall", "f1_score", "roc_auc"]].style.format({
    "accuracy": "{:.2%}", "precision": "{:.2%}", "recall": "{:.2%}",
    "f1_score": "{:.2%}", "roc_auc": "{:.4f}"
}))

summary = {
    "dataset": "GenImage official benchmark",
    "label_mapping": {"nature/real": 0, "ai-generated": 1},
    "generator_partition": {
        "training_generators": TRAINING_GENERATORS,
        "real_training_generator": REAL_TRAIN_GENERATOR,
        "validation_generator": VALIDATION_GENERATOR,
        "test_generator": TEST_GENERATOR,
    },
    "requested_images_per_class": {
        "train": TRAIN_PER_CLASS,
        "validation": VALIDATION_PER_CLASS,
        "test": TEST_PER_CLASS,
    },
    "exact_sha256_duplicates_removed_across_splits": True,
    "split_sizes": {
        "train": len(train_frame),
        "validation": len(validation_frame),
        "test": len(test_frame),
    },
    "starting_cnn_checkpoint": CNN_CHECKPOINT.name,
    "cnn_checkpoint_selection": cnn_selection,
    "starting_vit_checkpoint_dir": str(VIT_CHECKPOINT_DIR),
    "vit_checkpoint_selection": vit_selection,
    "original_huggingface_model": MODEL_ID,
    "original_huggingface_revision": MODEL_REVISION,
    "results": {
        "cifake_custom_cnn": cnn_results,
        "huggingface_vit": vit_results,
    },
    "warning": (
        "Test accuracy measures transfer to the held-out Midjourney generator only. "
        "Exact byte-identical duplicates were excluded, but visually similar or "
        "re-encoded duplicates may remain. This is not a universal real-world accuracy "
        "estimate; evaluate on additional unseen generators and independent real sources."
    ),
}
(METRICS_DIR / "ai_vs_real_finetuning_comparison.json").write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8"
)
comparison.to_csv(METRICS_DIR / "ai_vs_real_finetuning_comparison.csv", index=False)
print(f"Metrics saved under {METRICS_DIR}")
print(f"CNN checkpoint: {CNN_OUTPUT_DIR / 'cifake_custom_cnn_ai_vs_real_genimage_v3.keras'}")
print(f"ViT model + processor: {VIT_OUTPUT_DIR}")

## 7. Export AI-vs-real checkpoints and metrics

The output directory contains the new CNN `.keras` model, the ViT model plus processor files, and metrics. It is separate from the v2 inputs. Download the output ZIP and extract it under `artifacts/cifake/` only after reviewing the held-out-generator test results. The app will not switch to these checkpoints automatically.

In [ ]:
from google.colab import files
import shutil

OUTPUT_ARCHIVE = Path("/content/ai_vs_real_genimage_outputs_v3.zip")
if OUTPUT_ARCHIVE.exists():
    OUTPUT_ARCHIVE.unlink()
shutil.make_archive(str(OUTPUT_ARCHIVE.with_suffix("")), "zip", OUTPUT_DIR)
print(f"Created {OUTPUT_ARCHIVE} ({OUTPUT_ARCHIVE.stat().st_size / (1024**2):.1f} MiB)")
files.download(str(OUTPUT_ARCHIVE))

In [75]:
!7z x "{ADM_ZIP}" -o/content/genimage_extracted/ADM -y "imagenet_ai_0508_adm/val/*"
!7z x "{MJ_ZIP}"  -o/content/genimage_extracted/Midjourney -y "imagenet_midjourney/val/*"


7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan /content/drive/.shortcut-targets-by-id/1jGt10bwTbhEZuGXLyvrCuxOI0cBqQ1FS/genimage/ADM/                                                                                                1 file, 184678028 bytes (177 MiB)

Extracting archive: /content/drive/.shortcut-targets-by-id/1jGt10bwTbhEZuGXLyvrCuxOI0cBqQ1FS/genimage/ADM/imagenet_ai_0508_adm.zip
  1% 4096 Open               22% 73728 Open                45% 151552 Open                 67% 225280 Open                 86% 286720 Open                